# 03 - PLM masked-marginal scoring (Rep 1/3/4)

Scores ESM-2 650M, ESM-1v (env `esm`), and ESM C 600M (env `esmc`) by masking each of the 263 WT positions. Derives Rep 1 (masked-marginal scalar), Rep 3 (per-position surprisal vector, 20), and Rep 4 (site surprisal plus one-hot substitution, 40). Cached to `data/interim/`.

Standalone-runnable, shells to the correct conda env per model.

In [ ]:
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/".projectroot").exists())
sys.path.insert(0, str(_root))
from paths import RAW, INTERIM, PROCESSED, FIGURES, TABLES, PROJECT_ROOT
SRC = _root / "src"


In [ ]:
import subprocess, sys
def run_in(env, script, *args):
    """Run src/<script> in conda env <env>; stream output; raise on failure."""
    cmd = ["conda","run","--no-capture-output","-n",env,"python",str(SRC/script), *map(str,args)]
    print(">>", " ".join(cmd))
    r = subprocess.run(cmd)
    if r.returncode != 0: raise RuntimeError(f"{script} failed in env {env} (rc={r.returncode})")


### First, reconstruct the 263-aa mature WT (needed by every scorer)

In [ ]:
import pandas as pd, json, numpy as np
df=pd.read_parquet(PROCESSED/'tem1_firnberg_processed.parquet')
pool=df[~df.excluded_wetlab_validation]
positions=sorted(pool.position_linear.unique())
wt={}
for _,r in pool.iterrows(): wt.setdefault(r.position_linear, r.wt_aa)
wt_seq=''.join(wt[p] for p in positions)
assert len(wt_seq)==263 and wt_seq[:7]=='HPETLVK', wt_seq[:7]
json.dump({'wt_seq':wt_seq,'positions':positions}, open(INTERIM/'wt_seq.json','w'))
print('WT', len(wt_seq),'aa, starts', wt_seq[:12])

### Score each PLM, skipping anything already cached. ESM-2/ESM-1v run in `esm`, ESM C in `esmc`.

In [ ]:
for tag,(env,model,layer) in {
    'esm2_650m':('esm','esm2_t33_650M_UR50D',33),
    'esm1v':('esm','esm1v_t33_650M_UR90S_1',33),
}.items():
    if (INTERIM/f'mm_{tag}.npz').exists(): print('cached', tag); continue
    run_in(env,'score_mm.py',model,layer,tag)
if not (INTERIM/'mm_esmc.npz').exists(): run_in('esmc','score_esmc.py')
else: print('cached esmc')

### Derive Rep 1/3/4 and report zero-shot Spearman(scalar, DMS)

In [ ]:
from scipy.stats import spearmanr
for tag,name in [('esm2_650m','ESM-2 650M'),('esm1v','ESM-1v'),('esmc','ESM C 600M')]:
    run_in('betalactam-v2','derive_feats.py',tag)
